# Supply your own training and testing data

## Objective

Save the exact stages you choose, run one model and understand the result.
This lesson is independent of notebooks 2 through 6. Use a fresh kernel, the installed
package and Java 17. All data is invented. The temporary files are removed at the end.

Training is the history the model may learn from. Testing is the later period used to
check its predictions. Validation is an optional earlier check; omitting it does not
cause AutoForecast to make a split for you.

## Step 1: Start Spark
The next cell creates a small local workspace and enables progress messages.


In [ ]:
# Keep this lesson in a temporary folder and use two local Spark workers.
import datetime as dt
import tempfile
from pathlib import Path

from autoforecast import StudyRegistry, configure_logging, register_study
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

workspace = tempfile.TemporaryDirectory(prefix="autoforecast-lesson-")
work = Path(workspace.name)
spark = (SparkSession.builder.master("local[2]")
         .config("spark.sql.shuffle.partitions", "2")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
configure_logging()


### Understand the output

Spark is ready. The workspace belongs to this lesson, so it cannot replace another run.

## Step 2: Make a small dataset
We use two products and nine dates. Product B has no record on 1 January.


In [ ]:
# Make two products. Product B starts one day later than A.
observations = spark.createDataFrame([
    (product, dt.date(2024, 1, day), float(day))
    for product in ("A", "B")
    for day in range(1 if product == "A" else 2, 10)
], "product string, date date, units double")
observations.orderBy("product", "date").show()


### Understand the output

Each row is one product on one date. `units` is the value we want to predict.
B starts on 2 January; we do not add an earlier zero for it.

## Step 3: Choose where studies are saved
This is a complete registry-only configuration. A study configuration contains more:
its columns, dates, source and scoring rules. See notebook 2 for that route.


In [ ]:
# A registry file says where to save studies. It does not describe the data.
import yaml

registry_path = work / "registry.yaml"
registry_path.write_text(yaml.safe_dump({"storage": {"root": str(work / "registry")}}))
registry = StudyRegistry.from_config(registry_path, spark=spark)
print(registry.root)


### Understand the output

The printed path is storage, not a registered study. `scratch_root` defaults to its
`scratch` folder. On a cluster, both paths must be accessible to all Spark workers.
`from_config` uses the file; `from_environment` reads the documented environment settings.

## Step 4: Supply training and testing
The model may learn from 1 through 7 January. We check predictions for 8 and 9 January.
The ranges include both endpoints. They must be in time order and follow the chosen
frequency, with no gaps or overlaps between stages. For daily data, a stage ending
on 7 January is followed by one starting on 8 January. Validation, if supplied, sits
between training and testing. Each input must contain only rows within its own
stage range. We supply no validation data.


In [ ]:
# Filter each input ourselves; registration never silently removes out-of-range rows.
training = observations.filter(F.col("date") <= dt.date(2024, 1, 7))
testing = observations.filter(F.col("date") >= dt.date(2024, 1, 8))

# These arguments describe the scientific question and the missing-value rules.
settings = dict(
    registry=registry, series_id=["product"], date_col="date", target_col="units",
    target_unit="units", frequency="D", metric="mae",
    history_missing="preserve", actual_missing="error",
)
study_id = register_study(
    training_data=training, testing_data=testing,
    training_date_range=("2024-01-01", "2024-01-07"),
    testing_date_range=("2024-01-08", "2024-01-09"), **settings,
)
print(study_id)


### Understand the output

The ID identifies the captured data and rules. AutoForecast checks both products in
both stages, rejects duplicate dates and saves a frozen copy. Changing the source later
cannot change this saved study. Each lazy DataFrame is captured once; this is not a
transaction across unrelated DataFrames.

## Step 5: Check what was saved
`None` means validation was not configured. It is not a zero score or an empty stage.


In [ ]:
# Load the saved study without reading the original DataFrames again.
with registry.load_study(study_id) as study:
    print("Training:", study.training_date_range)
    print("Validation:", study.validation_date_range)
    print("Final fit:", study.refit_date_range)
    print("Testing:", study.testing_date_range)
    assert study.validation_data is None
    assert study.training_date_range == study.refit_date_range


### Understand the output

Training and final-fit ranges match. There is no separate validation fit.

## Step 6: Run one included model
Seasonal naive repeats an earlier value. With season length 1 it repeats the last
training value. MAE is the average size of the prediction error, in units.


In [ ]:
# Fit once on training data and score the two planned testing dates.
from autoforecast import run_study

result = run_study(study_id, "seasonal_naive", registry=registry)
print("Validation:", result.validation_metrics)
print("Testing:", result.testing_metrics)
print(result.summary())


### Understand the output

Validation is `None`. The model predicts 7 units for each product on both testing dates.
The actual values are 8 and 9, so the four absolute errors are 1, 2, 1 and 2.
Their average is `(1 + 2 + 1 + 2) / 4 = 1.5`: the MAE is **1.5 units**.
Lower MAE means smaller errors on the same observations. The summary includes two products
and four scored observations. These invented values teach the calculation; they are not a
real model comparison.

The other printed metrics describe different aspects of the same errors. RMSE gives larger
errors more weight. WAPE compares total absolute error with total actual units. Bias is the
average signed error; a negative value means predictions are too low. See notebook 5 for
more about results. Publication and MLflow export include only configured stages. No tuning runs unless you request it. If you need tuning,
declare `tuning_windows` or `tuning_schedule` before registration and follow notebook 4.
The inner tuning targets must stay inside training history.

## Step 7: Add validation when you want it
Here we shorten training to 1 through 5 January and use 6 and 7 January for validation.
Testing stays on 8 and 9 January. The final fit may use training plus validation.


In [ ]:
# Validation data and its range must always be supplied together.
with_validation = register_study(
    training_data=observations.filter(F.col("date") <= dt.date(2024, 1, 5)),
    validation_data=observations.filter(F.col("date").between(
        dt.date(2024, 1, 6), dt.date(2024, 1, 7))),
    testing_data=testing,
    training_date_range=("2024-01-01", "2024-01-05"),
    validation_date_range=("2024-01-06", "2024-01-07"),
    testing_date_range=("2024-01-08", "2024-01-09"), **settings,
)
print("Different study:", with_validation != study_id)


### Understand the output

The ID changes because the stage definitions changed. A run of this study fits for
validation, then refits on training plus validation for testing. An empty validation
DataFrame is an error. To omit validation, omit both arguments.

A planned horizon does not shrink when actuals arrive late. For example, a 14-day testing
range still requests 14 predictions when only 10 days have arrived. Choose `actual_missing`
explicitly: `error` stops, `mask` scores available observations where the metric permits it,
and `zero` treats missing observations as zero. None of these creates an omitted stage.
A known product with no actuals in a nonempty stage follows the same policy. Unexpected new
products still fail. An entirely empty supplied stage always fails in this release.

## Delta tables and features
On a Delta-capable Spark session, a stage can be `DeltaTable(name="catalog.schema.testing", version=42)`.
Import it from `autoforecast`. Omit the version to pin the latest version once. References
to the same physical table must agree on the version. The whole table must fit the stage's
range; filter a DataFrame yourself if you need a subset. A DataFrame created with
`spark.table()` is recorded as a DataFrame, without a claimed Delta version.

Extra columns are not automatically predictors. Features known in advance need both a
column role and evidence of when the value became known. In the example below, a planned
price was already known on 31 December. `exposure_col` names the observation-status column.
We use the string `exact` for measured targets in this example. Declaring a feature does not
make every model use it. Pick a model that supports that feature role.

Other existing roles include `static` for category columns that do not change per product,
`static_reals` for fixed numeric columns, and `past_only` for values observed only in history.
Direct inputs must already be prepared. Notebook 11 explains preparation hooks and gives
complete WRMSSE `references` settings for users who need weighted scoring.

## Step 8: Declare a known future feature
We add one price, its availability date and an observation status, then register the stages. The status `exact` means each target is a measured value.


In [ ]:
# A future feature needs its own declared availability date, not a future target value.
featured = (observations.withColumn("planned_price", F.lit(2.0))
            .withColumn("price_known_on", F.lit(dt.date(2023, 12, 31)))
            .withColumn("observation_status", F.lit("exact")))
feature_study = register_study(
    training_data=featured.filter(F.col("date") <= dt.date(2024, 1, 7)),
    testing_data=featured.filter(F.col("date") >= dt.date(2024, 1, 8)),
    training_date_range=("2024-01-01", "2024-01-07"),
    testing_date_range=("2024-01-08", "2024-01-09"),
    feature_roles={"known_future": ["planned_price"], "exposure_col": "observation_status"},
    feature_availability={"planned_price": "price_known_on"}, **settings,
)
# Check the saved declaration without starting another model fit.
with registry.load_study(feature_study) as study:
    print("Known future features:", study.panel.known_future)
    print("Availability columns:", study.recipe["feature_availability"])


### Understand the output

The saved feature is `planned_price`; its availability column is `price_known_on`.
The library checks availability at each fit boundary. It does not treat arbitrary extra
columns as model inputs or let testing targets become training history.

## Step 9: Finish the lesson
Remove only this lesson's temporary workspace after closing Spark.


In [ ]:
# These outputs were temporary examples. Save real studies in durable shared storage.
spark.stop()
workspace.cleanup()


### Understand the output

The lesson is complete. Use notebooks 5 through 9 to inspect or publish saved real runs.
